<a href="https://colab.research.google.com/github/NLSCMAP/EXCEL-DATA-PROCESSING/blob/main/%E3%80%8CExcel%E8%B3%87%E6%96%99%E8%87%AA%E5%8B%95%E5%8C%96%E7%B5%B1%E8%A8%88%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import os
import re
import glob
import warnings

# 忽略 pandas 的未來警告 (例如 to_datetime 的 UserWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# 嘗試匯入 tabula-py 和 xlsxwriter，這在 Colab 環境中是必要的
try:
    import tabula
    print("tabula-py 模組已載入。")
except ImportError:
    print("tabula-py 模組未安裝，將嘗試安裝...")
    try:
        os.system('apt-get update -qq && apt-get install openjdk-17-jre -y -qq')
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
        print("Java Runtime Environment (JRE) 安裝完成。")
        os.system('pip install tabula-py')
        import tabula
    except Exception as e:
        print(f"Java 或 tabula-py 安裝失敗，無法處理 PDF 檔案。錯誤: {e}")
        tabula = None

try:
    import xlsxwriter
    print("xlsxwriter 模組已載載。")
except ImportError:
    print("xlsxwriter 模組未安裝，將嘗試安裝...")
    os.system('pip install xlsxwriter')
    import xlsxwriter

# ====================================================================
# 【 全 域 設 定 】
# ====================================================================

# 輸入資料夾名稱：所有原始 Excel 和 PDF 檔案都必須放在此資料夾內
INPUT_FOLDER_NAME = 'input_files'
# 輸出報告檔案名稱
OUTPUT_FILE_NAME = '案件報告統計表_合併批次處理.xlsx'

# 報告客製化資訊
REPORT_CONFIGURATION = {
    "REPORT_NAME": "Ekran System側錄監控統計表",
    "REPORT_TIME_RANGE": "114年8月份", # 統計期間：請自行修改
    "REPORT_DATE": "114年9月22日", # 建置日期：請自行修改
    "SYSTEM_NAME": "跳板機群組", # 系統別：請自行修改
    "MONITORING_SERVER": "監控跳板機", # 監控跳板機：請自行修改
}

# *** 核心變動: 只需要這兩個欄位 ***
REQUIRED_COLS = [
    '使用者名稱',         # 對應 UserClient Identifier (Client name/User name)
    '活動時間'            # 對應 Activity Time (時長字串, e.g., '1h 16m 8s')
]

# PDF 表格掃描區域 (Top, Left, Bottom, Right) - 調整 Top 跳過報告頂部的 Details/Filter 文字區塊
PDF_AREA = [180, 0, 800, 612] # 假設 A4 頁面寬度 612 點

# ====================================================================
# 【 輔 助 函 數 】
# ====================================================================

def convert_seconds_to_dhms(total_seconds):
    """將總秒數轉換為 '0d 0h 0m 0s' 格式"""
    if pd.isna(total_seconds) or total_seconds <= 0:
        return '0d 0h 0m 0s'

    total_seconds = int(total_seconds)
    days = total_seconds // (24 * 3600)
    total_seconds %= (24 * 3600)
    hours = total_seconds // 3600
    total_seconds %= 3600
    minutes = total_seconds // 60
    seconds = total_seconds % 60

    return f"{days}d {hours}h {minutes}m {seconds}s"

def convert_dhms_to_seconds(dhms_str):
    """將 '0d 0h 0m 0s' 或 '1h 16m 8s' 格式的時長字串轉換為總秒數"""
    if pd.isna(dhms_str) or not isinstance(dhms_str, str):
        return 0

    # 清理並統一格式，確保空格分隔
    dhms_str = dhms_str.replace('\n', ' ').strip()
    total_seconds = 0

    # 匹配所有數字及其後面的單位 (d, h, m, s)
    matches = re.findall(r'(\d+)\s*([dhms])', dhms_str, re.IGNORECASE)

    for value, unit in matches:
        value = int(value)
        unit = unit.lower()
        if unit == 'd':
            total_seconds += value * 86400
        elif unit == 'h':
            total_seconds += value * 3600
        elif unit == 'm':
            total_seconds += value * 60
        elif unit == 's':
            total_seconds += value

    return total_seconds

# ====================================================================
# 【 主 要 處 理 函 數 】
# ====================================================================

def read_file_to_df(filepath):
    """根據檔案類型讀取檔案並返回 DataFrame"""
    filename = os.path.basename(filepath)
    print(f"--- 正在處理檔案: {filename} ---")

    df = pd.DataFrame()
    N_REQUIRED_COLS = len(REQUIRED_COLS)

    try:
        if filepath.lower().endswith(('.xlsx', '.xls')):
            # *** 處理 Excel 檔案 (核心精進: 讀取所有工作表，強制映射前 2 欄) ***

            # 使用 header=None 讀取所有工作表，保留原始結構
            all_sheets = pd.read_excel(filepath, sheet_name=None, header=None)

            df_list = []
            for sheet_name, sheet_df in all_sheets.items():
                print(f"DEBUG: 正在處理工作表: {sheet_name}")
                # 假設實際數據從第 6 行開始 (索引 5)，用來跳過報表頂部標頭
                if sheet_df.shape[0] >= 6:

                    df_data = sheet_df[5:].copy()

                    # --- 關鍵步驟: 僅取前 N 欄並強制命名 ---
                    if df_data.shape[1] >= N_REQUIRED_COLS:
                        # 選擇前 N 欄 (即前 2 欄)
                        df_temp = df_data.iloc[:, :N_REQUIRED_COLS].copy()
                        # 強制命名欄位為目標名稱 ['使用者名稱', '活動時間']
                        df_temp.columns = REQUIRED_COLS
                        df_list.append(df_temp)
                    else:
                        print(f"警告: 工作表 '{sheet_name}' 實際欄位數 ({df_data.shape[1]}) 少於所需欄位數 ({N_REQUIRED_COLS})，跳過。")
                else:
                    print(f"警告: 工作表 '{sheet_name}' 行數不足 (小於 6 行數據)，跳過。")


            if df_list:
                df = pd.concat(df_list, ignore_index=True)
                print(f"檔案讀取與合併完成。總行數: {len(df)}")
                print(f"Excel 欄位已透過位置 (前 {N_REQUIRED_COLS} 欄) 映射完成。")
                print(f"DEBUG: Found columns in DF: {df.columns.tolist()}")

            else:
                print(f"檔案 {filename} 讀取失敗或內容為空。跳過。")
                return None

        elif filepath.lower().endswith('.pdf'):
            # 處理 PDF 檔案 (依賴欄位順序)
            if tabula is None:
                print("錯誤: tabula-py 模組未能成功載入，無法處理 PDF 檔案。")
                return None

            print(f"嘗試從 PDF 檔案 {filename} 提取表格...")

            dfs = tabula.read_pdf(
                filepath,
                pages='all',
                stream=True,
                area=PDF_AREA,
                pandas_options={'header': None}
            )

            if not dfs:
                print(f"錯誤: 從 PDF 檔案 {filename} 提取表格失敗。未找到任何表格。")
                return None

            df = pd.concat(dfs, ignore_index=True)
            print(f"PDF 提取完成。總行數: {len(df)}")

            # === PDF 欄位數必須至少匹配 N 欄 ===
            N_COLS_ACTUAL = df.shape[1]

            if N_COLS_ACTUAL >= N_REQUIRED_COLS:
                print(f"偵測到 {N_COLS_ACTUAL} 個欄位，將取前 {N_REQUIRED_COLS} 欄進行映射。")

                # 進行命名和過濾
                col_rename_map = {i: col_name for i, col_name in enumerate(REQUIRED_COLS)}

                actual_cols_to_rename = {k: v for k, v in col_rename_map.items() if k < N_COLS_ACTUAL}
                df.rename(columns=actual_cols_to_rename, inplace=True)

                # 僅保留我們需要的 N 欄
                df = df.reindex(columns=REQUIRED_COLS)
                print(f"PDF 欄位手動命名和過濾完成。保留欄位: {REQUIRED_COLS}")

                print(f"DEBUG: Found columns in DF: {df.columns.tolist()}")

            else:
                print(f"錯誤: PDF 提取的欄位數 ({N_COLS_ACTUAL}) 少於所需欄位數 ({N_REQUIRED_COLS})。無法處理。")
                return None

        else:
            print(f"警告: 檔案 {filename} 格式不受支援。跳過。")
            return None

    except Exception as e:
        print(f"處理檔案 {filename} 過程中發生錯誤: {e}")
        return None

    # --- 共同後續處理 (數據清洗邏輯) ---
    if df.empty:
        print(f"檔案 {filename} 讀取失敗或內容為空。跳過。")
        return None

    initial_row_count = len(df)
    print(f"DEBUG: 1. 初始行數: {initial_row_count}")

    # 確保主要識別欄位填充成功 (處理 PDF 稀疏數據)
    df['使用者名稱'] = df['使用者名稱'].ffill()

    # 1. 剔除 活動時間 缺失或看起來像日期/時間的行
    df_clean = df.copy()

    # 清理 Activity Time 的字串格式
    df_clean['活動時間'] = df_clean['活動時間'].astype(str).str.replace(r'[^\x00-\x7F]+', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()

    # 關鍵修正: 如果 活動時間 包含日期時間分隔符 (/, -, :), 則視為無效時長 (避免欄位錯位)
    date_like_filter = df_clean['活動時間'].str.contains(r'[/:-]', na=False)

    if date_like_filter.any():
        print(f"DEBUG: 偵測到 {date_like_filter.sum()} 行 活動時間 包含日期/時間標記，將其視為無效數據。")
        df_clean.loc[date_like_filter, '活動時間'] = np.nan

    df_clean.dropna(subset=['活動時間'], how='any', inplace=True)

    print(f"DEBUG: 1.2. 剔除 活動時間 缺失或日期/時間後行數: {len(df_clean)}")

    # 2. 活動時間 字串轉換為總秒數 (Total Seconds)
    df_clean['Total Seconds'] = df_clean['活動時間'].apply(convert_dhms_to_seconds)

    # 3. 剔除 Total Seconds <= 0 的行
    rows_lost_in_seconds_filter = len(df_clean)
    df_clean = df_clean[df_clean['Total Seconds'] > 0]
    rows_lost_in_seconds_filter -= len(df_clean)

    if rows_lost_in_seconds_filter > 0 and len(df_clean) == 0:
        sample_times = df['活動時間'].head(5).tolist() if initial_row_count > 0 else []
        print(f"警告: {filename} 100% 的 活動時間 轉換為 0 秒。請檢查原始數據格式是否為 DHMS。")
        print(f"DEBUG: 原始 活動時間 範例 (處理前): {sample_times}")
        return None

    # 4. 建立新的分組鍵：清理 使用者名稱 (移除後面的時長字串, 例如 ' 1h 16m 8s')
    def clean_user_client_identifier(identifier_str):
        if pd.isna(identifier_str) or not isinstance(identifier_str, str):
            return identifier_str

        cleaned = re.sub(r'(\s+[\d\s]*[dhms]\s*)+$', '', identifier_str, flags=re.IGNORECASE).strip()

        return cleaned.strip()

    df_clean['Group Key'] = df_clean['使用者名稱'].apply(clean_user_client_identifier)

    # 5. 確保 Group Key 不為空
    df_clean.dropna(subset=['Group Key'], how='any', inplace=True)

    print(f"數據清洗完成。剔除不符合要求的行數: {initial_row_count - len(df_clean)}")

    # 6. 標記來源檔案 (用於多分頁輸出)
    df_clean['Source File'] = filename

    print(f"最終有效行數 (返回前): {len(df_clean)}")

    return df_clean

def generate_report(file_data_list, config):
    """根據每個檔案的數據列表生成 Excel 報告，每個檔案一個分頁"""

    if not file_data_list:
        print("沒有有效的數據可以生成報告。")
        return

    output_filepath = config['OUTPUT_FILE_NAME']

    # 設置 Writer 和 Workbook
    writer = pd.ExcelWriter(output_filepath, engine='xlsxwriter')
    workbook = writer.book

    # 定義格式
    bold_format = workbook.add_format({'bold': True, 'align': 'center', 'valign': 'vcenter', 'border': 1})
    center_format = workbook.add_format({'align': 'center', 'valign': 'vcenter', 'border': 1})
    left_format = workbook.add_format({'align': 'left', 'valign': 'vcenter', 'border': 1})
    title_format = workbook.add_format({'bold': True, 'align': 'left', 'valign': 'vcenter', 'font_size': 12})
    note_format = workbook.add_format({'text_wrap': True, 'align': 'left', 'valign': 'top'})

    # --- 處理每個檔案數據並創建分頁 ---
    for df_data in file_data_list:
        source_filename = df_data['Source File'].iloc[0]
        merged_df = df_data.copy()

        # 使用檔案名稱作為分頁名稱，並清理掉後綴 (只取前 31 個字元)
        sheet_name = source_filename.replace('附件-', '').replace('.pdf', '').replace('.xlsx', '').replace('.xls', '')
        sheet_name = sheet_name[:31]

        worksheet = workbook.add_worksheet(sheet_name)
        print(f"--- 正在生成分頁: {sheet_name} ---")

        # --- 1. 執行統計彙總 ---
        # 計算每個 Group Key (清理後的使用者識別碼) 的統計數據
        stats_grouped = merged_df.groupby('Group Key').agg(
            Count=('Total Seconds', 'count'),
            Total_Seconds=('Total Seconds', 'sum')
        ).reset_index()

        # 計算總時長 (DHMS 格式)
        stats_grouped['總時長'] = stats_grouped['Total_Seconds'].apply(convert_seconds_to_dhms)
        stats_grouped['次數'] = stats_grouped['Count'].astype(int)

        # 重新命名欄位以匹配報告需求
        stats_grouped.rename(columns={'Group Key': '使用者名稱'}, inplace=True)
        stats_df = stats_grouped[['使用者名稱', '次數', '總時長']] # 最終報告保留三欄

        # --- 寫入標題和元數據 ---

        worksheet.merge_range('A1:C1', config['REPORT_NAME'], title_format)

        row = 1
        col_width = [15, 25]

        worksheet.write(row, 0, '統計期間', left_format)
        worksheet.write(row, 1, config['REPORT_TIME_RANGE'], left_format)
        worksheet.write(row, 2, '建置日期', left_format)
        worksheet.write(row, 3, config['REPORT_DATE'], left_format)

        worksheet.set_column(0, 0, col_width[0])
        worksheet.set_column(1, 1, col_width[1])
        worksheet.set_column(2, 2, col_width[0])
        worksheet.set_column(3, 3, col_width[1])


        worksheet.write(row + 1, 0, '系統別', left_format)
        worksheet.write(row + 1, 1, config['SYSTEM_NAME'], left_format)
        worksheet.write(row + 1, 2, '資料來源', left_format)
        worksheet.write(row + 1, 3, source_filename, left_format)


        # 寫入總時長
        total_seconds_all = merged_df['Total Seconds'].sum()
        total_time_dhms = convert_seconds_to_dhms(total_seconds_all)
        worksheet.write(row + 2, 0, '總時長 (所有用戶)', left_format)
        worksheet.merge_range(row + 2, 1, row + 2, 3, total_time_dhms, center_format) # 合併總時長儲存格

        # --- 寫入統計表頭 (精簡為三欄) ---

        start_row = 6

        worksheet.write(start_row - 1, 0, '使用者名稱', bold_format)
        worksheet.write(start_row - 1, 1, '次數', bold_format)
        worksheet.write(start_row - 1, 2, '總時長', bold_format)

        worksheet.set_column(0, 0, 25) # 使用者名稱欄位寬度加寬
        worksheet.set_column(1, 1, 10, center_format)
        worksheet.set_column(2, 2, 20, center_format)

        # --- 寫入數據內容 ---

        for row_idx, row_data in stats_df.iterrows():
            worksheet.write(start_row + row_idx, 0, row_data['使用者名稱'], left_format)
            worksheet.write(start_row + row_idx, 1, row_data['次數'], center_format)
            worksheet.write(start_row + row_idx, 2, row_data['總時長'], center_format)

        # --- 寫入底部備註 ---

        note_start_row = start_row + len(stats_df) + 2
        note_text = (
            "(1).連線次數及使用時長皆包含測試連線(可能僅連線數秒)\n"
            "(2).用戶說明(我在自己擴充)\n"
            f"(3).本頁數據僅來自檔案: {source_filename}"
        )

        worksheet.merge_range(
            note_start_row, 0, note_start_row + 2, 3,
            note_text, note_format
        )

    # 關閉並保存檔案
    writer.close()

    # --- 輸出總結 ---
    total_files = len(file_data_list)
    total_records = sum(len(df) for df in file_data_list)

    print(f"\n✅ 統計報告已成功生成至: {output_filepath}")
    print(f"   報告包含 {total_files} 個分頁，總共處理了 {total_records} 筆有效記錄。")

# ====================================================================
# 【 主 程式 執 行 區 塊 】
# ====================================================================

if __name__ == "__main__":

    if not os.path.isdir(INPUT_FOLDER_NAME):
        os.makedirs(INPUT_FOLDER_NAME)
        print(f"請將原始檔案 (Excel/PDF) 上傳到新建立的資料夾: '{INPUT_FOLDER_NAME}' 中。")
    else:
        all_files = glob.glob(os.path.join(INPUT_FOLDER_NAME, '*.xlsx')) + \
                    glob.glob(os.path.join(INPUT_FOLDER_NAME, '*.xls')) + \
                    glob.glob(os.path.join(INPUT_FOLDER_NAME, '*.pdf'))

        if not all_files:
            print(f"錯誤: 在資料夾 '{INPUT_FOLDER_NAME}' 中未找到任何 Excel (.xlsx/.xls) 或 PDF (.pdf) 檔案。")
        else:
            print(f"找到 {len(all_files)} 個檔案，開始批次處理...")

            file_data_list = []

            for filepath in all_files:
                df_clean = read_file_to_df(filepath)
                if df_clean is not None and not df_clean.empty:
                    file_data_list.append(df_clean)

            if not file_data_list:
                print("所有檔案處理後皆無有效數據。報告生成失敗。")
            else:
                report_config = REPORT_CONFIGURATION.copy()
                report_config['OUTPUT_FILE_NAME'] = OUTPUT_FILE_NAME

                generate_report(file_data_list, report_config)

tabula-py 模組未安裝，將嘗試安裝...
Java Runtime Environment (JRE) 安裝完成。
xlsxwriter 模組未安裝，將嘗試安裝...
請將原始檔案 (Excel/PDF) 上傳到新建立的資料夾: 'input_files' 中。
